# Synthetic Rx Rebate Data — Exploration

This notebook walks through the synthetic pharmaceutical rebate dataset:
- **Claims**: 500K prescription fills across retail, mail, and specialty channels
- **Drugs**: 300 NDC records with specialty classification
- **Formulary**: Tier placement and utilization management flags per client
- **Contracts**: Rebate agreements with 4 basis types
- **Invoices**: Quarterly rebate invoices with ±5% noise (normal) and injected anomalies

The data is generated by `src/synthetic_data_gen/` and saved to `data/synthetic/*.parquet`.

**Purpose**: Understand distributions before building anomaly detection models.

## 1. Setup & Data Load

In [ ]:
import polars as pl
import numpy as np
import altair as alt
from pathlib import Path

In [ ]:
# Altair renderer — works in Jupyter and VS Code

# alt.renderers.enable("default")
alt.renderers.enable("vegafusion")

In [ ]:
# Point to the synthetic data directory
# Adjust this path if running the notebook from a different working directory
data_dir = Path("../data/synthetic")

claims    = pl.read_parquet(data_dir / "claims.parquet")
drugs     = pl.read_parquet(data_dir / "drugs.parquet")
formulary = pl.read_parquet(data_dir / "formulary.parquet")
contracts = pl.read_parquet(data_dir / "contracts.parquet")
invoices  = pl.read_parquet(data_dir / "invoices_with_anomalies.parquet")

print("Data loaded successfully.")

## 2. Dataset Overview

In [ ]:
# Row counts across all tables
print(f"Claims    : {len(claims):>10,}  (individual prescription fills)")
print(f"Drugs     : {len(drugs):>10,}  (unique NDC-11 records)")
print(f"Formulary : {len(formulary):>10,}  (NDC × client tier assignments)")
print(f"Contracts : {len(contracts):>10,}  (rebate agreements)")
print(f"Invoices  : {len(invoices):>10,}  (quarterly rebate invoices)")
print()

# Key column snapshots
print("Claims columns :", claims.columns)
print("Invoices columns:", invoices.columns)

## 3. Claims: Dispensing Channel Distribution

Three channels exist in this dataset:
- **Retail** (~70%): traditional pharmacy fills
- **Mail** (~20%): 90-day mail-order supplies
- **Specialty** (~10%): high-cost injectable / biologic therapies

Channel matters for rebate contracts because some manufacturers exclude certain channels.

In [ ]:
channel_counts = (
    claims
    .group_by("channel")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("pct")
    )
    .sort("count", descending=True)
)

print(channel_counts)

chart = (
    alt.Chart(channel_counts.to_pandas())
    .mark_bar()
    .encode(
        x=alt.X("channel:N", sort="-y", title="Channel"),
        y=alt.Y("count:Q", title="Number of Claims"),
        color=alt.Color("channel:N", legend=None),
        tooltip=["channel", "count", "pct"],
    )
    .properties(title="Claims by Dispensing Channel", width=400, height=300)
)
chart

## 4. Claims: Days Supply Distribution

Days supply is the number of days a prescription fill covers.  Common values:
- **30 days**: standard retail fill
- **60 days**: extended retail fill
- **84 days**: some mail-order programs
- **90 days**: standard mail-order / specialty fill

Rebate contracts denominate utilization in "30-day equivalents", so a 90-day fill = 3 units.

In [ ]:
days_counts = (
    claims
    .group_by("days_supply")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("pct")
    )
    .sort("days_supply")
)

print(days_counts)

chart = (
    alt.Chart(days_counts.to_pandas())
    .mark_bar()
    .encode(
        x=alt.X("days_supply:O", title="Days Supply"),
        y=alt.Y("count:Q", title="Number of Claims"),
        tooltip=["days_supply", "count", "pct"],
    )
    .properties(title="Claims by Days Supply", width=350, height=280)
)
chart

## 5. Claims: Gross Drug Cost by Channel

Specialty drugs are typically 10–30× more expensive than retail drugs.  
This cost difference is why specialty rebates are high-value targets for recovery.

In [ ]:
cost_stats = (
    claims
    .group_by("channel")
    .agg([
        pl.col("gross_drug_cost").mean().alias("mean"),
        pl.col("gross_drug_cost").median().alias("median"),
        pl.col("gross_drug_cost").quantile(0.25).alias("p25"),
        pl.col("gross_drug_cost").quantile(0.75).alias("p75"),
        pl.col("gross_drug_cost").max().alias("max"),
    ])
    .sort("mean", descending=True)
)
print(cost_stats)

# Box-plot approximation: mean ± IQR bars
cost_pd = cost_stats.to_pandas()

bars = (
    alt.Chart(cost_pd)
    .mark_bar()
    .encode(
        x=alt.X("channel:N", title="Channel"),
        y=alt.Y("p25:Q", title="Gross Drug Cost ($)"),
        y2="p75:Q",
        color="channel:N",
        tooltip=["channel", "mean", "median", "p25", "p75"],
    )
)

medians = (
    alt.Chart(cost_pd)
    .mark_tick(color="black", thickness=2, size=20)
    .encode(
        x=alt.X("channel:N"),
        y=alt.Y("median:Q"),
    )
)

(bars + medians).properties(
    title="Gross Drug Cost by Channel — IQR bars, median tick",
    width=350, height=300
)

## 6. Claims: Fill Date Trend Over Time

The synthetic dataset spans 2024-01-01 to 2025-12-31 with roughly uniform monthly volume.  
In real data you would see seasonal peaks (e.g. January new deductible, flu season).

In [ ]:
monthly = (
    claims
    .with_columns(
        pl.col("fill_date").dt.strftime("%Y-%m").alias("month")
    )
    .group_by("month")
    .agg(pl.len().alias("claims"))
    .sort("month")
)

chart = (
    alt.Chart(monthly.to_pandas())
    .mark_line(point=True)
    .encode(
        x=alt.X("month:O", title="Month", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("claims:Q", title="Number of Claims", scale=alt.Scale(zero=False)),
        tooltip=["month", "claims"],
    )
    .properties(title="Monthly Claim Volume (2024–2025)", width=650, height=280)
)
chart

## 7. Drugs: Specialty vs. Non-Specialty

`specialty_flag = True` means the drug is a high-cost injectable or biologic.  
All injectable drugs are forced to specialty channel.  
~30% of the 300 NDCs are specialty — they generate a disproportionate share of rebate dollars.

In [ ]:
spec_counts = (
    drugs
    .group_by("specialty_flag")
    .agg(pl.len().alias("count"))
    .with_columns(
        pl.col("specialty_flag").cast(pl.Utf8).alias("label"),
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("pct"),
    )
)
print(spec_counts)

chart = (
    alt.Chart(spec_counts.to_pandas())
    .mark_arc()
    .encode(
        theta=alt.Theta("count:Q"),
        color=alt.Color("label:N", title="Specialty"),
        tooltip=["label", "count", "pct"],
    )
    .properties(title="Drug Specialty Classification (300 NDCs)", width=300, height=300)
)
chart

## 8. Formulary: Tier Distribution

Formulary tiers control member cost-sharing:
- **Tier 1**: Generic — lowest copay
- **Tier 2**: Preferred brand
- **Tier 3**: Non-preferred brand
- **Tier 4**: Specialty preferred
- **Tier 5**: Specialty non-preferred
- **Tier 6**: Exclusion tier — not covered

Manufacturers pay higher rebates to secure lower (preferred) tier placement.

In [ ]:
tier_counts = (
    formulary
    .group_by("tier")
    .agg(pl.len().alias("count"))
    .sort("tier")
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("pct")
    )
)
print(tier_counts)

tier_labels = {1: "T1-Generic", 2: "T2-Pref Brand", 3: "T3-Non-Pref",
               4: "T4-Spec Pref", 5: "T5-Spec Non-Pref", 6: "T6-Excluded"}

tier_pd = tier_counts.to_pandas()
tier_pd["tier_label"] = tier_pd["tier"].map(tier_labels)

chart = (
    alt.Chart(tier_pd)
    .mark_bar()
    .encode(
        x=alt.X("tier_label:N", sort=None, title="Formulary Tier"),
        y=alt.Y("count:Q", title="Records"),
        color=alt.Color("tier:O", scale=alt.Scale(scheme="blues"), legend=None),
        tooltip=["tier_label", "count", "pct"],
    )
    .properties(title="Formulary Records by Tier", width=450, height=300)
)
chart

## 9. Contracts: Rebate Basis Distribution

Four rebate basis types are modeled:
- **PER_30_DAY_SCRIPT**: fixed dollar amount per 30-day-equivalent fill
- **PERCENT_GROSS_COST**: percentage of the drug's gross cost
- **PER_UNIT**: rebate per dispensed unit (tablet, mL, etc.)
- **PMPM_GUARANTEE**: per-member-per-month minimum guarantee

Each basis requires different utilization data when auditing invoices.

In [ ]:
basis_counts = (
    contracts
    .group_by("rebate_basis")
    .agg(pl.len().alias("count"))
    .with_columns(
        (pl.col("count") / pl.col("count").sum() * 100).round(1).alias("pct")
    )
    .sort("count", descending=True)
)
print(basis_counts)

chart = (
    alt.Chart(basis_counts.to_pandas())
    .mark_bar()
    .encode(
        x=alt.X("rebate_basis:N", sort="-y", title="Rebate Basis"),
        y=alt.Y("count:Q", title="Number of Contracts"),
        color=alt.Color("rebate_basis:N", legend=None),
        tooltip=["rebate_basis", "count", "pct"],
    )
    .properties(title="Contracts by Rebate Basis Type", width=420, height=300)
)
chart

## 10. Invoices: Rebate Realization Ratio

**Rebate realization** = `actual_rebate / expected_rebate`

In a healthy invoice, noise is small (±5%). Values far from 1.0 signal potential leakage:
- `< 0.90`: manufacturer significantly underpaid
- `= 0.00`: rebate completely missing (non-payment or missing invoice)
- `> 1.05`: overpayment (rare; could be error or retroactive adjustment)

In [ ]:
# Only rows where expected_rebate is positive
inv_nonzero = invoices.filter(pl.col("expected_rebate") > 0)

realization = inv_nonzero.with_columns(
    (pl.col("actual_rebate") / pl.col("expected_rebate")).alias("realization")
)

print(f"Rows with expected_rebate > 0 : {len(inv_nonzero):,}")
print()
print("Realization distribution:")
print(realization["realization"].describe())

# Clamp to [0, 1.1] for visualization clarity
hist_data = realization.with_columns(
    pl.col("realization").clip(0.0, 1.1)
).to_pandas()

chart = (
    alt.Chart(hist_data)
    .mark_bar()
    .encode(
        x=alt.X("realization:Q", bin=alt.Bin(maxbins=50), title="Realization Ratio"),
        y=alt.Y("count():Q", title="Number of Invoice Rows"),
        tooltip=["count()"],
    )
    .properties(
        title="Rebate Realization Distribution — normal ≈ 0.95–1.05 (capped at 1.1)",
        width=600, height=300,
    )
)
chart

## 11. Invoices: Top 20 Products by Expected Rebate

Anomaly detection audits are most valuable when focused on high-dollar products.  
This chart shows which NDC × manufacturer combinations have the largest expected rebate.

In [ ]:
top_products = (
    invoices
    .group_by(["ndc11", "manufacturer"])
    .agg(pl.col("expected_rebate").sum().alias("total_expected_rebate"))
    .sort("total_expected_rebate", descending=True)
    .head(20)
    .with_columns(
        (pl.col("ndc11") + " / " + pl.col("manufacturer")).alias("product")
    )
)

print("Top 20 products by total expected rebate:")
print(top_products[["product", "total_expected_rebate"]])

chart = (
    alt.Chart(top_products.to_pandas())
    .mark_bar()
    .encode(
        y=alt.Y("product:N", sort="-x", title="NDC / Manufacturer"),
        x=alt.X("total_expected_rebate:Q", title="Total Expected Rebate ($)"),
        color=alt.Color("manufacturer:N", legend=None),
        tooltip=["product", "total_expected_rebate"],
    )
    .properties(title="Top 20 Products by Expected Rebate", width=500, height=450)
)
chart

## 12. Anomaly Patterns: Rule-Based Summary

Because `anomaly_labels.parquet` contains the schema definition (empty when no anomalies were persisted),  
we reconstruct ground-truth labels directly from `invoices_with_anomalies.parquet`  
by applying the same detection rules used during injection.

Known injected anomaly types:
| Type | Description |
|------|-------------|
| MISSING_REBATE | expected > 0, actual = 0, ndc in drug master |
| UNMAPPED_NDC | ndc not in drug master (synthetic NDC added during injection) |
| REBATE_YIELD_COLLAPSE | realization < 50% with non-zero actual |
| DISPUTE_SPIKE | disputed_rebate > 0 (clustered at manufacturer+brand+quarter level) |

In [ ]:
drug_ndcs = set(drugs["ndc11"].to_list())

# MISSING_REBATE: expected > 0, actual = 0, NDC exists in drug master
missing_rebate = invoices.filter(
    (pl.col("expected_rebate") > 0)
    & (pl.col("actual_rebate") == 0)
    & pl.col("ndc11").is_in(list(drug_ndcs))
)

# UNMAPPED_NDC: NDC does not exist in the drug master
unmapped_ndc = invoices.filter(
    ~pl.col("ndc11").is_in(list(drug_ndcs))
    & (pl.col("expected_rebate") > 0)
)

# REBATE_YIELD_COLLAPSE: realization < 50% with non-trivial expected rebate
yield_collapse = invoices.filter(
    (pl.col("expected_rebate") > 10)
    & (pl.col("actual_rebate") > 0)
    & ((pl.col("actual_rebate") / pl.col("expected_rebate")) < 0.50)
)

# DISPUTE_SPIKE: rows where disputed_rebate > 0
dispute_brands = (
    invoices
    .filter(pl.col("disputed_rebate") > 0)
    .group_by(["manufacturer", "brand_family", "invoice_quarter"])
    .agg(pl.col("disputed_rebate").sum().alias("total_disputed"))
    .sort("total_disputed", descending=True)
)

print("=== Detected Anomaly Patterns ===")
print(f"MISSING_REBATE rows         : {len(missing_rebate)}")
print(f"UNMAPPED_NDC rows           : {len(unmapped_ndc)}")
print(f"REBATE_YIELD_COLLAPSE rows  : {len(yield_collapse)}")
print(f"DISPUTE_SPIKE brand-quarters: {len(dispute_brands)}")
print()
print("Missing rebate detail:")
print(missing_rebate[["invoice_quarter", "ndc11", "client_id", "expected_rebate"]])
print()
print("Dispute spike summary:")
print(dispute_brands)

## 13. Referential Integrity Check

Before training models, verify that keys are consistent across tables.  
Foreign-key violations would indicate data quality issues (or intentional unmapped-NDC anomalies).

In [ ]:
claim_ndcs   = set(claims["ndc11"].to_list())
invoice_ndcs = set(invoices["ndc11"].to_list())

# Check 1: All claim NDCs exist in drug master
orphan_claim_ndcs = claim_ndcs - drug_ndcs
status_1 = "PASS" if not orphan_claim_ndcs else f"FAIL — {len(orphan_claim_ndcs)} orphan NDCs"
print(f"Claim NDCs in drug master     : {status_1}")

# Check 2: Invoice NDCs either in drug master OR are injected unmapped NDCs
orphan_invoice_ndcs = invoice_ndcs - drug_ndcs
status_2 = (
    "PASS (expected — these are injected UNMAPPED_NDC anomalies)"
    if orphan_invoice_ndcs
    else "PASS"
)
print(f"Invoice NDCs not in drug master: {len(orphan_invoice_ndcs)} → {status_2}")
if orphan_invoice_ndcs:
    print(f"  Unmapped NDCs: {orphan_invoice_ndcs}")

# Check 3: Formulary NDCs in drug master
formulary_ndcs = set(formulary["ndc11"].to_list())
orphan_formulary = formulary_ndcs - drug_ndcs
status_3 = "PASS" if not orphan_formulary else f"FAIL — {len(orphan_formulary)} orphan NDCs"
print(f"Formulary NDCs in drug master : {status_3}")

# Check 4: Invoice quarters are well-formed
malformed_quarters = invoices.filter(
    ~pl.col("invoice_quarter").str.contains(r"^\d{4}-Q[1-4]$")
)
status_4 = "PASS" if len(malformed_quarters) == 0 else f"FAIL — {len(malformed_quarters)} rows"
print(f"Invoice quarter format        : {status_4}")

## 14. Summary

This notebook explored the full synthetic rebate dataset.  
Key takeaways for anomaly detection:

In [ ]:
total_expected = invoices["expected_rebate"].sum()
total_actual   = invoices["actual_rebate"].sum()
rebate_gap     = total_expected - total_actual

print("""
=== Synthetic Dataset Overview ===

Tables & Rows
  Claims    : 500,000  (2024-01-01 – 2025-12-31, 3 channels)
  Drugs     :     300  (30% specialty / 70% non-specialty)
  Formulary :  10,500  (6 tiers, PA/ST/QL flags)
  Contracts :  10,870  (4 rebate basis types)
  Invoices  : 118,124  (8 quarters × NDC × client, with anomalies)

Normal Behavior
  Rebate realization (actual/expected) peaks at 1.0 ± 5% noise
  Specialty drugs are ~30x more expensive than retail
""")

print(f"Total expected rebate : ${total_expected:>15,.2f}")
print(f"Total actual rebate   : ${total_actual:>15,.2f}")
print(f"Overall rebate gap    : ${rebate_gap:>15,.2f}")
print()
print("""
Injected Anomalies (ground truth recovered from data)
  MISSING_REBATE        — expected > 0, actual = 0
  UNMAPPED_NDC          — NDC not in drug master
  REBATE_YIELD_COLLAPSE — realization < 50%
  DISPUTE_SPIKE         — disputed_rebate > 0

Next Step → notebooks/02_train_anomaly_models.ipynb
""")